In [1]:
! pip install torch torchvision

In [2]:
import torch
from torch import nn
import numpy as np
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torchvision.models import vit_b_16, ViT_B_16_Weights
import os

In [3]:
if torch.cuda.is_available():
    dev = "cuda:0"
elif torch.backends.mps.is_available():
    dev = "mps"
else:
    dev = "cpu"
device = torch.device(dev)
device

device(type='mps')

In [4]:
weights = ViT_B_16_Weights.IMAGENET1K_V1

preprocess = weights.transforms()

In [5]:
train_dir = 'dogs-vs-cats/train/'
dog_dir = train_dir + 'dog/'
cat_dir = train_dir + 'cat/'
os.makedirs(dog_dir, exist_ok=True)
os.makedirs(cat_dir, exist_ok=True)

In [6]:
for filename in os.listdir(train_dir):
    if 'dog.' in filename.lower():
        os.rename(os.path.join(train_dir, filename), os.path.join(dog_dir, filename))
    if 'cat.' in filename.lower():
        os.rename(os.path.join(train_dir, filename), os.path.join(cat_dir, filename))

In [7]:
train_ds = ImageFolder(root=train_dir, transform=preprocess)

In [8]:
train_ds, valid_ds = torch.utils.data.random_split(train_ds, [0.8, 0.2])

In [9]:
mini_batch_size = 64
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=mini_batch_size, shuffle=True, drop_last=False, num_workers=4)
valid_dl = torch.utils.data.DataLoader(valid_ds, batch_size=mini_batch_size, num_workers=4)

In [10]:
model = vit_b_16(weights=weights)
model.heads.head = torch.nn.Linear(model.heads.head.in_features, 2)
model = model.to(device)
model

VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_a

In [11]:
class WrappedDataLoader:
    def __init__(self, dl, func):
        self.dl = dl
        self.func = func

    def __len__(self):
        return len(self.dl)

    def __iter__(self):
        for b in self.dl:
            yield (self.func(*b))


def put_to_gpu(x, y):
    return x.to(device), y.to(device)

In [12]:
for param in model.parameters():
    param.requires_grad = False
for param in model.heads.head.parameters():
    param.requires_grad = True

In [13]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

1538

In [14]:
optimizer = torch.optim.Adam(model.parameters())

In [15]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_model_state = None

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

    def load_best_model(self, model):
        model.load_state_dict(self.best_model_state)

In [16]:
early_stopping = EarlyStopping(patience=3, delta=0.01)

In [17]:
def fit(epochs, model, optimizer, train_dl, valid_dl=None):
    loss_func = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()

        for X_mb, y_mb in train_dl:
            y_hat = model(X_mb)

            loss = loss_func(y_hat, y_mb)
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

        model.eval()
        with torch.no_grad():
            train_loss = sum(loss_func(model(X_mb), y_mb) for X_mb, y_mb in train_dl)
            valid_loss = sum(loss_func(model(X_mb), y_mb) for X_mb, y_mb in valid_dl)
        print('epoch {}, training loss {}'.format(epoch + 1, train_loss / len(train_dl)))
        print('epoch {}, validation loss {}'.format(epoch + 1, valid_loss / len(valid_dl)))

        early_stopping(valid_loss, model)
        if early_stopping.early_stop:
            print("Early stopping")
            break

    print('Finished training')

    return model

In [18]:
epochs = 10

model = fit(epochs, model, optimizer, WrappedDataLoader(train_dl, put_to_gpu), WrappedDataLoader(valid_dl, put_to_gpu))

epoch 1, training loss 0.021304810419678688
epoch 1, validation loss 0.022546589374542236
epoch 2, training loss 0.0163879357278347
epoch 2, validation loss 0.019275717437267303
epoch 3, training loss 0.01383095420897007
epoch 3, validation loss 0.01894485019147396
epoch 4, training loss 0.012514191679656506
epoch 4, validation loss 0.018862318247556686
epoch 5, training loss 0.010856643319129944
epoch 5, validation loss 0.018891068175435066
epoch 6, training loss 0.009811796247959137
epoch 6, validation loss 0.01887129619717598
Early stopping
Finished training


In [19]:
early_stopping.load_best_model(model)
model = model.cpu()

In [20]:
def evaluate(model, data_loader):    
    model.eval()
    accuracy = 0
    with torch.no_grad():
        for X, y in data_loader:
            y_hat = model(X).cpu().numpy()
            y_hat = np.argmax(y_hat, axis=1)
            accuracy += (y_hat == y.cpu().numpy()).mean()
    accuracy /= len(data_loader)

    return accuracy

In [21]:
evaluate(model.to(device), WrappedDataLoader(train_dl, put_to_gpu))

np.float64(0.9974540734824281)

In [22]:
evaluate(model.to(device), WrappedDataLoader(valid_dl, put_to_gpu))

np.float64(0.9954509493670886)